# 🖃 StampVue - Google Colab 一鍵公開部署筆記本

本筆記本將 StampVue (印章拍照擷取與去背系統) 部署於 Google Colab 環境，並透過 **Cloudflare Tunnel** 免費產生安全的 **公開 HTTPS 網址**。

> 💡 **為什麼需要 HTTPS？**
> 瀏覽器的即時相機鏡頭拍攝 (`getUserMedia`) 依照 W3C 安全規範必須在 HTTPS 網域下才能授權使用。Cloudflare Tunnel 免費提供合格的 SSL 憑證，可供手機、平板與桌機直接開啟使用！

## 🚀 步驟 1: 下載並設定環境 (Node.js & Cloudflare Tunnel)

In [ ]:
# 1. 檢查並更新 Node.js 環境
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node -v
!npm -v

# 2. 下載 Cloudflare Tunnel CLI (用於產生免費公開 HTTPS 網址)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

## 📦 步驟 2: 取得 StampVue 原始碼

請選擇以下其中一種方式取得程式碼：
- **選項 A (推薦)**：直接使用您的 GitHub 倉庫網址進行 `git clone`
- **選項 B**：將本地端 StampVue 專案打包為 `StampVue.zip` 上傳至 Colab 左側檔案區後解壓縮

In [ ]:
import os

# 若已存在則先清理或跳過
if not os.path.exists('/content/StampVue'):
    # 【選項 A】請將此處替換為您的 GitHub 專案倉庫 URL (若為公開倉庫):
    # !git clone https://github.com/YOUR_GITHUB_USERNAME/StampVue.git /content/StampVue
    
    # 【選項 B】若您直接上傳了 StampVue.zip 到 Colab：
    if os.path.exists('/content/StampVue.zip'):
        !unzip -q /content/StampVue.zip -d /content/StampVue
        print("已成功解壓縮 StampVue.zip！")
    else:
        print("💡 請於左側檔案欄上傳 StampVue.zip，或取消註解上方 git clone 指令以取得程式碼！")
else:
    print("StampVue 專案資料夾已就緒！")

## ⚡ 步驟 3: 安裝相依套件並啟動服務與公開穿透網址

In [ ]:
import subprocess
import time
import re

# 切換到前端目錄
%cd /content/StampVue/frontend

# 安裝相依套件並建立生產建置
!npm install
!npm run build

print("✅ 前端套件安裝與建置完成！正在啟動靜態伺服器與 Cloudflare Tunnel...")

# 1. 背景啟動 Vite 預覽伺服器 (監聽 5173 port)
server_process = subprocess.Popen(
    ["npx", "vite", "preview", "--port", "5173", "--host", "0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(2)

# 2. 背景啟動 Cloudflare Tunnel 穿透 5173 port
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5173"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
start_time = time.time()

# 從 cloudflared 輸出中抓取 https://*.trycloudflare.com
while time.time() - start_time < 30:
    line = tunnel_process.stderr.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    from IPython.display import display, HTML
    html_code = f"""
    <div style='background: linear-gradient(135deg, #1e293b, #0f172a); border: 2px solid #ef4444; border-radius: 12px; padding: 20px; color: white; font-family: sans-serif; max-width: 600px; margin: 15px 0;'>
        <h3 style='margin-top: 0; color: #f87171;'>🎉 StampVue 已成功公開發布！</h3>
        <p style='margin-bottom: 12px;'>已透過 Cloudflare 免費 HTTPS 穿透通道公開上線，支援即時視訊相機拍攝與智慧去背：</p>
        <p style='font-size: 1.15rem; font-weight: bold;'>
            👉 <a href='{public_url}' target='_blank' style='color: #60a5fa; text-decoration: underline;'>{public_url}</a>
        </p>
        <p style='font-size: 0.85rem; color: #94a3b8; margin-bottom: 0;'>💡 請在任何手機、平板或電腦瀏覽器點擊上方連結即可使用！</p>
    </div>
    """
    display(HTML(html_code))
else:
    print("⚠️ 未能在 30 秒內取得 Cloudflare Tunnel 網址，請檢查記錄：")
    print(tunnel_process.stderr.read())

# 保持 Cell 運行
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("服務已停止。")
    server_process.kill()
    tunnel_process.kill()